# 08 · xgboost · TUNED_V2

Chạy từ trên xuống. Kiểm tra Hardware Check trước khi bật START_TRAINING trong ô cuối. Mỗi modality nạp cấu hình từ tuning_lock.json đã xác minh.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, subprocess, uuid
DRIVE_FOLDER_ID = os.environ["PHISHSHIELD_DRIVE_FOLDER_ID"]
DRIVE_ROOT = None
RUNTIME = Path("/content/phishshield-ai-run")
# Nếu dùng shortcut, chọn đường dẫn nhìn thấy trong bảng Files của Colab.
candidate = "/content/drive/MyDrive/phishshield-ai"  # @param {type:"string"}
if not candidate or not Path(candidate).resolve().is_relative_to(Path("/content/drive")):
    raise ValueError("Chọn folder đã mount. Có thể thêm shortcut vào My Drive trước.")
MODEL_NAME = "xgboost"
DEVICE = "cpu"
EXPERIMENT_PHASE = "TUNED_V2"
MODALITIES = ["url_only", "url_html"]
TRAIN_SEEDS = [42, 43, 44]


### Xác minh Folder ID

Notebook đối chiếu metadata của mount. Nếu mount thiếu metadata, Drive API tạo một tệp thử trong folder có đúng ID và đối chiếu nội dung qua filesystem. Nếu xác minh thất bại, kiểm tra quyền Editor và shortcut rồi chạy lại.

In [ ]:
def resolve_drive_root(candidate=None, service=None):
    candidate = candidate or os.environ.get("PHISHSHIELD_DRIVE_ROOT")
    if not candidate:
        raise ValueError("Chọn đường dẫn mount thực tế hoặc shortcut trong My Drive.")
    root = Path(candidate).resolve(strict=True)
    if not root.is_dir():
        raise ValueError("Cần chọn thư mục Drive.")
    try:
        actual = (Path(str(root) + ":user.drive.id").read_text() if os.name == "nt"
                  else os.getxattr(root, "user.drive.id").decode())
    except (AttributeError, OSError):
        actual = None
    if actual == DRIVE_FOLDER_ID:
        return root
    if not service:
        raise ValueError("Chưa xác minh được Folder ID. Cấp quyền Drive API trong notebook.")
    # Prove the mount points to the API parent by reading a unique API-created file.
    from googleapiclient.http import MediaInMemoryUpload
    import time
    files = service.files()
    meta = files.get(fileId=DRIVE_FOLDER_ID, fields="mimeType,capabilities(canAddChildren)",
                     supportsAllDrives=True).execute()
    if meta["mimeType"] != "application/vnd.google-apps.folder" or not meta["capabilities"]["canAddChildren"]:
        raise PermissionError("Cần quyền ghi trong folder chính thức.")
    key = ".phishshield-probe-" + uuid.uuid4().hex
    value = os.urandom(32)
    probe = files.create(body={"name": key, "parents": [DRIVE_FOLDER_ID]},
                         media_body=MediaInMemoryUpload(value), supportsAllDrives=True, fields="id").execute()
    try:
        for _ in range(30):
            if (root / key).is_file() and (root / key).read_bytes() == value:
                return root
            time.sleep(1)
        raise ValueError("Sai đường dẫn hoặc mount chưa đồng bộ. Kiểm tra shortcut trong My Drive.")
    finally:
        files.delete(fileId=probe["id"], supportsAllDrives=True).execute()

try:
    DRIVE_ROOT = resolve_drive_root(candidate)
except ValueError:
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    creds, _ = google.auth.default()
    DRIVE_ROOT = resolve_drive_root(candidate, build("drive", "v3", credentials=creds))
print("Workspace đã xác minh:", DRIVE_ROOT)

In [ ]:
# Chạy một bản sao mã nguồn trong runtime. Mọi bản ghi bền vững đi vào folder đã xác minh.
import importlib.util
spec = importlib.util.spec_from_file_location("verified_storage", DRIVE_ROOT / "src/phishshield/utils/storage.py")
storage = importlib.util.module_from_spec(spec)
spec.loader.exec_module(storage)
storage.validate_drive_access(DRIVE_ROOT)
storage.sync_source_to_runtime(DRIVE_ROOT, RUNTIME)
os.environ.update(PHISHSHIELD_DRIVE_ROOT=str(DRIVE_ROOT), PHISHSHIELD_RUNTIME_ROOT=str(RUNTIME))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv==0.11.11"], check=True)
if DEVICE != "cpu":
    raise ValueError("Gate V2 yêu cầu cpu cho xgboost.")
extras = ['trees']
args = [part for extra in extras for part in ("--extra", extra)]
env = os.environ.copy()
venv = RUNTIME / (".venv-cuda-mlp" if DEVICE == "cuda" else ".venv")
env["UV_PROJECT_ENVIRONMENT"] = str(venv)
subprocess.run([sys.executable, "-m", "uv", "sync", "--project", str(RUNTIME),
                "--python", "3.11", "--frozen", "--inexact", *args], env=env, check=True)
python = str(venv / "bin/python")

In [ ]:

from pathlib import Path

path = DRIVE_ROOT / "data/processed/cohort.csv"

print("Đường dẫn:", path)
print("File tồn tại:", path.exists())

if path.exists():
    print("Dung lượng:", path.stat().st_size, "bytes")


In [ ]:
# Chạy script tiền xử lý dữ liệu để tạo tệp cohort.csv
# Đảm bảo bạn đã chạy đầy đủ các ô cell-01, cell-03, cell-04 trước khi chạy ô này
import subprocess
import sys
from pathlib import Path

# Khôi phục biến 'python' và 'RUNTIME' nếu chưa chạy các ô trước
try:
    python_cmd = python
except NameError:
    python_cmd = str(Path("/content/phishshield-ai-run/.venv/bin/python"))

try:
    runtime_path = RUNTIME
except NameError:
    runtime_path = Path("/content/phishshield-ai-run")

try:
    print("--- Đang tiến hành tiền xử lý dữ liệu ---")
    # Chạy module tiền xử lý dữ liệu của phishshield
    subprocess.run(
        [python_cmd, "-m", "phishshield.data.process"],
        check=True
    )
    print("✅ Tiền xử lý dữ liệu hoàn tất! Hãy kiểm tra xem file cohort.csv đã xuất hiện chưa.")
except subprocess.CalledProcessError as e:
    # Nếu không có module trên, thử chạy trực tiếp file script trong thư mục scripts
    try:
        script_path = runtime_path / "scripts/process.py"
        if script_path.exists():
            subprocess.run([python_cmd, str(script_path)], check=True)
            print("✅ Chạy script process.py hoàn tất!")
        else:
            print("❌ Không tìm thấy script tiền xử lý mặc định. Hãy kiểm tra lại thư mục gốc hoặc tài liệu dự án để biết file chạy tiền xử lý tên là gì.")
    except Exception as ex:
        print(f"❌ Gặp lỗi khi chạy tiền xử lý: {ex}")

In [ ]:
subprocess.run([python, "-m", "phishshield.utils.hardware", MODEL_NAME, DEVICE], check=True)

In [ ]:
START_TRAINING = True  # @param {type:"boolean"}
if START_TRAINING:
    if EXPERIMENT_PHASE != "TUNED_V2" or MODEL_NAME != "xgboost":
        raise ValueError("Sai phase/model cho notebook này.")
    command = [python, str(RUNTIME / "scripts/workflow.py"), "8", "--device", DEVICE]
    for modality in MODALITIES:
        for seed in TRAIN_SEEDS:
            try:
                # Thêm capture_output=True để có thể in chi tiết lỗi nếu script con bị crash
                subprocess.run(
                    [*command, "--phase", EXPERIMENT_PHASE, "--seed", str(seed), "--mode", modality],
                    check=True,
                    capture_output=True,
                    text=True
                )
            except subprocess.CalledProcessError as e:
                print(f"❌ Lỗi xảy ra khi chạy: {modality} - Seed: {seed}")
                print("--- STDOUT ---")
                print(e.stdout)
                print("--- STDERR ---")
                print(e.stderr)
                raise e
else:
    print("TUNED_V2 chưa bắt đầu.")